## WEEK - 7 Basic RAG System


In [ ]:
%pip list

In [ ]:
%pip install \
transformers==4.41.2 \
sentence-transformers==2.7.0 \
torch \
faiss-cpu \
langchain \
langchain-community \
langchain-core \
langchain-text-splitters \
beautifulsoup4 \
requests

In [ ]:
# ==============================
# 1. IMPORTS
# ==============================
import requests
from bs4 import BeautifulSoup

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from sentence_transformers import CrossEncoder
from transformers import pipeline

# ==============================
# 2. DATA INGESTION
# ==============================
urls = [
    "https://docs.crawl4ai.com/core/simple-crawling/",
    "https://cohere.com/llmu/dense-retrieval",
    "https://docs.crawl4ai.com/core/crawler-result/",
    "https://docs.crawl4ai.com/core/browser-crawler-config/",
]


def fetch_docs(url):
    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    elements = soup.find_all(["p", "li", "code", "pre", "h1", "h2", "h3"])

    text = "\n".join([el.get_text(separator=" ") for el in elements])
    return text


def build_documents(urls):
    docs = []
    for url in urls:
        content = fetch_docs(url)
        docs.append(Document(page_content=content, metadata={"source": url}))
    return docs


raw_docs = build_documents(urls)

# ==============================
# 3. CHUNKING
# ==============================
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

chunked_docs = []
for doc in raw_docs:
    chunks = splitter.split_documents([doc])
    chunked_docs.extend(chunks)

# ==============================
# 4. EMBEDDINGS + VECTOR STORE
# ==============================
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunked_docs, embeddings)


# ==============================
# 5. RETRIEVAL
# ==============================
def retrieve(query, k=10):
    return vectorstore.similarity_search(query, k=k)


# ==============================
# 6. FILTERING (OPTIONAL)
# ==============================
def filter_docs(docs, keyword=None):
    if not keyword:
        return docs

    return [d for d in docs if keyword.lower() in d.metadata.get("source", "").lower()]


# ==============================
# 7. RE-RANKING ⭐
# ==============================
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query, docs, top_k=3):
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)

    scored_docs = list(zip(docs, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    return [doc for doc, _ in scored_docs[:top_k]]


# ==============================
# 8. CONTEXT BUILDER
# ==============================
def build_context(docs):
    return "\n\n".join([d.page_content for d in docs])


# ==============================
# 9. LOCAL LLM (FREE) ⭐ (GENERATOR)
# ==============================
generator = pipeline(
    "text2text-generation",  # better for QA
    model="google/flan-t5-base",
    max_new_tokens=256,
)


def generate_answer(query, context):
    prompt = f"""
    Answer the question using ONLY the context below.
    If answer is not in context, say "I don't know".

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    result = generator(prompt)
    return result[0]["generated_text"]


# ==============================
# 10. FULL PIPELINE 🚀
# ==============================
def rag_pipeline(query):

    # Step 1: Retrieve
    retrieved_docs = retrieve(query, k=10)

    # Step 2: Filter (optional)
    filtered_docs = filter_docs(retrieved_docs)

    # Step 3: Re-rank
    reranked_docs = rerank(query, filtered_docs, top_k=3)

    # Step 4: Build context
    context = build_context(reranked_docs)

    # Step 5: Generate answer
    answer = generate_answer(query, context)

    return answer

/home/shrutik/Srutik/12-Week-Internship/.rag/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
# ==============================
# 11. TEST
# ==============================
query = "What is pricing of Gemini?"

response = rag_pipeline(query)

print("\nQ:", query)
print("\nA:", response)


Q: What is pricing of Gemini?

A: I don't know


In [4]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [14]:
%pip install chromadb

  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 7.9 MB/s  0:00:02m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 8.3 MB/s  0:00:00 eta 0:00:01
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 9.4 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 12.4 MB/s  

In [2]:
%pip install -qU langchain-huggingface sentence-transformers


Note: you may need to restart the kernel to use updated packages.


In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

/home/shrutik/Srutik/12-Week-Internship/.rag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
files=[
    ("/home/shrutik/Downloads/aiml.pdf"),
    ("/home/shrutik/Downloads/health.pdf")
]
all_docs = []

In [29]:
for file in files:
    loader = PyPDFLoader(file)
    docs = loader.load()
all_docs.extend(docs)

In [49]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap =150
)

In [50]:
chunked_docs = splitter.split_documents(all_docs)

In [51]:
embeddings = HuggingFaceEmbeddings(model="all-miniLM-L6-V2")

/home/shrutik/Srutik/12-Week-Internship/.rag/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [52]:
db = Chroma.from_documents(chunked_docs,embeddings,collection_name="multifile_rag")

In [69]:
query = " fatigue?"

results = db.similarity_search_with_score(query, k=3)

from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

pairs = [[query, doc.page_content] for doc, _ in results]
scores = reranker.predict(pairs)

best_idx = scores.argmax()
best_score = scores[best_idx]

if best_score < 0.5:
    print("I don't know")
else:
    print(results[best_idx][0].page_content)

fatigue and health issues.
Exercise: Exercise is physical activity that improves strength, endurance, and overall health. 
Regular activities like walking, running, and strength training enhance cardiovascular fitness and 
muscle development.
Sleep: Sleep is a natural restorative process essential for physical and mental health. Adequate sleep
(7–9 hours) improves memory, mood, and body recovery. Poor sleep can lead to stress and reduced 
productivity.


In [ ]:
# for i in results:
#     print("-"*100)
#     print(i.page_content)

----------------------------------------------------------------------------------------------------
fatigue and health issues.
Exercise: Exercise is physical activity that improves strength, endurance, and overall health. 
Regular activities like walking, running, and strength training enhance cardiovascular fitness and 
muscle development.
Sleep: Sleep is a natural restorative process essential for physical and mental health. Adequate sleep
(7–9 hours) improves memory, mood, and body recovery. Poor sleep can lead to stress and reduced 
productivity.
----------------------------------------------------------------------------------------------------
fatigue and health issues.
Exercise: Exercise is physical activity that improves strength, endurance, and overall health. 
Regular activities like walking, running, and strength training enhance cardiovascular fitness and 
muscle development.
Sleep: Sleep is a natural restorative process essential for physical and mental health. Adequate s

In [58]:
query = "what is ML?"

results = db.similarity_search_with_relevance_scores(query, k=3)

for doc, score in results:
    print("Score:", score)
    print(doc.page_content[:200])
    print("-"*50)

Score: 0.051299222366699815
AI & ML Concepts (Optimized for Similarity Search)
Large Language Models (LLMs): Large Language Models are deep learning models trained on
massive text datasets. They understand and generate human-lik
--------------------------------------------------
Score: -0.07616979056334516
Deep Learning: Deep learning is a subset of machine learning using neural networks with multiple
layers. Applied in NLP, vision, and speech tasks.
Machine Learning: Machine learning is a field of AI w
--------------------------------------------------
Score: -0.26000161541961453
Proper chunking improves retrieval accuracy. Includes fixed-size and semantic chunking strategies.
Retrieval-Augmented Generation (RAG): RAG combines vector search with language models to
retrieve rel
--------------------------------------------------


/tmp/ipykernel_289422/3320614347.py:3: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'keywords': '', 'subject': '(unspecified)', 'author': '(anonymous)', 'creationdate': '2026-03-26T06:19:31+00:00', 'producer': 'ReportLab PDF Library - (opensource)', 'page': 0, 'moddate': '2026-03-26T06:19:31+00:00', 'title': '(anonymous)', 'source': '/home/shrutik/Downloads/aiml.pdf', 'trapped': '/False', 'total_pages': 1, 'page_label': '1', 'creator': '(unspecified)'}, page_content='AI & ML Concepts (Optimized for Similarity Search)\nLarge Language Models (LLMs): Large Language Models are deep learning models trained on\nmassive text datasets. They understand and generate human-like language. Used in chatbots,\nsummarization, and question answering.\nEmbeddings: Embeddings convert text into numerical vectors that capture semantic meaning.\nSimilar texts produce similar vectors. Used in semantic search, clustering, and recommendations.'), 0.051299222366699815), (Docum